**Install**

In [1]:
!pip install transformers

**Set-up**

In [2]:
from transformers import pipeline

llm = pipeline("text-generation", model="distilgpt2")

**Data**

In [3]:
RESUMES = {
    "Strong Candidate": """
    Data Scientist with 5 years experience.
    Skills: Python, Machine Learning, Deep Learning, NLP, SQL.
    Tools: TensorFlow, PyTorch, Pandas.
    """,

    "Average Candidate": """
    Data Analyst with 2 years experience.
    Skills: Python, Excel, SQL.
    Tools: Pandas, Power BI.
    """,

    "Weak Candidate": """
    Fresher with basic knowledge.
    Skills: Excel.
    """
}

JOB_DESCRIPTION = """
Looking for Data Scientist.
Required Skills: Python, Machine Learning, NLP, SQL.
Experience: 3+ years.
"""

**Pipeline Function**

In [4]:
def generate(prompt):
    output = llm(
        prompt,
        max_new_tokens=60,
        do_sample=False
    )
    return output[0]['generated_text'].replace(prompt, "").strip()

In [5]:
from transformers import pipeline

# Load model
llm = pipeline("text-generation", model="google/flan-t5-base")

# Define generate function
def generate(prompt):
    return llm(prompt, max_length=200)[0]['generated_text']

JOB = """
Looking for Data Scientist.
Required Skills: Python, Machine Learning, NLP, SQL.
Experience: 3+ years.
"""

**Run Pipeline**

In [6]:
import re

def extract_info(resume):
    skills = []
    skill_keywords = ["python", "machine learning", "nlp", "sql", "excel", "deep learning"]

    for skill in skill_keywords:
        if skill.lower() in resume.lower():
            skills.append(skill)

    # extract experience
    exp_match = re.search(r"(\d+)\s*years", resume.lower())
    experience = int(exp_match.group(1)) if exp_match else 0

    return {
        "skills": skills,
        "experience": experience
    }


def match_profile(profile, job_skills):
    matched = [s for s in profile["skills"] if s in job_skills]
    missing = [s for s in job_skills if s not in profile["skills"]]

    return {
        "matched": matched,
        "missing": missing
    }


def score_candidate(match, experience):
    skill_score = (len(match["matched"]) / 4) * 70
    exp_score = min(experience / 5, 1) * 40

    score = int(skill_score + exp_score)
    score = min(score, 100)   # 👈 fix

    if score >= 75:
        level = "Strong"
    elif score >= 40:
        level = "Average"
    else:
        level = "Weak"

    return score, level

def explain(score, match):
    return (
        f"The candidate matches {len(match['matched'])} required skills "
        f"and lacks {len(match['missing'])}. "
        f"The score reflects both skill alignment and experience level."
    )

# JOB SKILLS
job_skills = ["python", "machine learning", "nlp", "sql"]


# RUN PIPELINE
for name, resume in RESUMES.items():
    print("\n==============================")
    print(f"Candidate: {name}")

    extracted = extract_info(resume)
    print("Extracted:", extracted)

    matched = match_profile(extracted, job_skills)
    print("Match:", matched)

    score, level = score_candidate(matched, extracted["experience"])
    print(f"Score: {score}/100 | Level: {level}")

    explanation = explain(score, matched)
    print("Explanation:", explanation)